# 논문용 bottle coreset ablation

MVTec AD train-normal에서 1%, 2.5%, 5%, 10%, 20% memory bank를 각각 독립 생성하고, 추적 가능한 Jetson bundle만 Google Drive에 저장합니다. Colab GPU 런타임에서 위에서 아래로 실행하세요.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import shutil, subprocess, sys

REPOSITORY_URL = 'https://github.com/minwoo1119/edge-anomaly-detection.git'
REPOSITORY_ROOT = Path('/content/edge-anomaly-detection')
DRIVE_INPUT_ROOT = Path('/content/drive/MyDrive/edge-anomaly-input')
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/edge-anomaly-output')
CATEGORY = 'bottle'
RATIOS = ['0.01', '0.025', '0.05', '0.10', '0.20']
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')

In [2]:
import torch
from google.colab import drive
drive.mount('/content/drive')
assert torch.cuda.is_available(), 'GPU 런타임을 선택하세요.'
print('GPU:', torch.cuda.get_device_name(0))

Mounted at /content/drive
GPU: Tesla T4


In [3]:
if (REPOSITORY_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPOSITORY_ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY_ROOT)], check=True)
%cd /content/edge-anomaly-detection
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

/content/edge-anomaly-detection


CompletedProcess(args=['git', 'rev-parse', 'HEAD'], returncode=0)

In [4]:
source_category = DRIVE_INPUT_ROOT / 'datasets' / 'MVTecAD' / CATEGORY
assert (source_category / 'train' / 'good').is_dir(), source_category
assert (source_category / 'test' / 'good').is_dir(), source_category
assert (source_category / 'ground_truth').is_dir(), source_category
local_dataset_root = Path('/content/MVTecAD')
local_category = local_dataset_root / CATEGORY
if not local_category.exists():
    local_dataset_root.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source_category, local_category)
print('dataset:', local_category)

dataset: /content/MVTecAD/bottle


In [5]:
local_output = Path('/content') / f'coreset_{CATEGORY}_{RUN_ID}'
command = [
    sys.executable, 'src/run_coreset_ablation.py',
    '--dataset-root', str(local_dataset_root),
    '--category', CATEGORY,
    '--output-root', str(local_output),
    '--ratios', *RATIOS,
    '--accelerator', 'gpu', '--devices', '1', '--seed', '42',
]
print(' '.join(command))
subprocess.run(command, check=True)

/usr/bin/python3 src/run_coreset_ablation.py --dataset-root /content/MVTecAD --category bottle --output-root /content/coreset_bottle_20261010T010021Z --ratios 0.01 0.025 0.05 0.10 0.20 --accelerator gpu --devices 1 --seed 42


CompletedProcess(args=['/usr/bin/python3', 'src/run_coreset_ablation.py', '--dataset-root', '/content/MVTecAD', '--category', 'bottle', '--output-root', '/content/coreset_bottle_20261010T010021Z', '--ratios', '0.01', '0.025', '0.05', '0.10', '0.20', '--accelerator', 'gpu', '--devices', '1', '--seed', '42'], returncode=0)

In [6]:
drive_output = DRIVE_OUTPUT_ROOT / f'coreset_{CATEGORY}_{RUN_ID}'
drive_output.mkdir(parents=True, exist_ok=False)
bundles = sorted(local_output.rglob(f'{CATEGORY}_artifacts.zip'))
assert len(bundles) == 5, bundles
for bundle in bundles:
    tag = bundle.parents[1].name
    target = drive_output / tag
    target.mkdir()
    subprocess.run([sys.executable, 'src/verify_artifact_bundle.py', '--bundle', str(bundle)], check=True)
    shutil.copy2(bundle, target / bundle.name)
    shutil.copy2(bundle.with_suffix(bundle.suffix + '.sha256'), target / (bundle.name + '.sha256'))
print('Drive output:', drive_output)
print('bundles:', [str(path.relative_to(drive_output)) for path in drive_output.rglob('*.zip')])

Drive output: /content/drive/MyDrive/edge-anomaly-output/coreset_bottle_20261010T010021Z
bundles: ['c001/bottle_artifacts.zip', 'c0025/bottle_artifacts.zip', 'c005/bottle_artifacts.zip', 'c01/bottle_artifacts.zip', 'c02/bottle_artifacts.zip']
